# Blue River mit RTC-Tools: Reservoir-Optimierung

## Lernziele

In diesem Notebook lernen Sie, wie Sie das **Blue River**-Beispiel aus `Inhalt/simple_reservoir` ausführen — plattformunabhängig (Windows, Linux, macOS) und ohne separate RTC-Backend-Umgebung:

- Das ShortTerm-Szenario starten und Ergebnisse prüfen.
- Ergebnisse für `TroutLake_V`, `TroutLake_Q_out` und `RiverCity_Q` mit Plotly visualisieren.
- Ziele (`goal_table.csv`) komfortabel über eine ipywidgets-GUI bearbeiten und speichern — ausführen und plotten geschieht dann in normalen Codezellen weiter unten.
- **Plots als PNG/HTML pro Lauf-Tag (`RUN_TAG`) exportieren**, um verschiedene Goal-Konfigurationen sauber zu vergleichen.
- Optional: LongTerm-Szenario ausführen und vergleichen.

> **Hinweis:** Die Optimierung läuft direkt in der Kursumgebung — `rtc-tools`, `rtc-tools-channel-flow`, `rtc-tools-interface` und `kaleido` sind Bestandteil von `requirements.txt`. Ein separates `venvRTC-Tools` ist nicht mehr nötig. Das Notebook ist damit auch in **Binder** und **Codespaces** lauffähig.


## Umgebung einrichten

- Das Notebook läuft in der normalen Kursumgebung (Conda/`requirements.txt`).
- Die **Eingangsdaten** liegen in `Inhalt/Notebook_Daten/RTC_BlueRiver`.
- **Ausgaben** werden in `Inhalt/Notebook_Daten/RTC_BlueRiver/runtime_case/output` geschrieben (CSV, Diagnostik, JSON-Cache, exportierte Figuren).

**Voraussetzungen:** `pip install -r requirements.txt` in der aktiven Conda-Umgebung. Damit sind alle nötigen Pakete (`rtc-tools` 2.7+, `casadi`, `pymoca`, `pandas`, `plotly`, `ipywidgets`, `kaleido`) vorhanden.

> **Optional, für fortgeschrittene Nutzung:** Wer das Modell in OMEdit visuell bearbeiten möchte, kann OpenModelica (https://openmodelica.org/) installieren — für das *Ausführen* des Notebooks ist OMEdit jedoch **nicht** erforderlich.


In [2]:
# Smoke-Check: läuft RTC-Tools im Kernel?
import sys, platform, importlib

print(f"Plattform:  {platform.system()} ({platform.machine()})")
print(f"Python:     {sys.version.split()[0]}")

missing = []
for pkg in ("rtctools", "casadi", "pymoca", "rtctools_interface",
            "pandas", "plotly", "ipywidgets"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    raise ImportError(
        "Fehlende Pakete: " + ", ".join(missing) +
        ". Bitte 'pip install -r requirements.txt' in der aktiven Umgebung ausführen."
    )

import rtctools, casadi
print(f"rtctools:   {rtctools.__version__}")
print(f"casadi:     {casadi.__version__}")
print("Alle Kernel-Pakete vorhanden — bereit für die Optimierung.")


Plattform:  Windows (AMD64)
Python:     3.10.16
rtctools:   2.7.3
casadi:     3.7.2
Alle Kernel-Pakete vorhanden — bereit für die Optimierung.


## Optional: Modell visuell bearbeiten (OpenModelica)

Wenn Sie das Modell **`Inhalt/simple_reservoir/BlueRiver2/model/BlueRiver.mo`** grafisch bearbeiten oder erweitern möchten, können Sie OpenModelica installieren — für das *Ausführen* der Optimierung ist das **nicht** notwendig.

1. OpenModelica (OMEdit) von https://openmodelica.org/ installieren.
2. OMEdit starten.
3. **Tools → Options → Libraries**: *Load latest Modelica version on startup* deaktivieren und **Modelica 3.2.3+maint.om** als Systembibliothek hinzufügen.
4. `BlueRiver.mo` öffnen und mit *Check Model* validieren.

Das Notebook selbst kompiliert und löst das Modell direkt über `rtc-tools` (CasADi-Backend, IPOPT-Solver) — ohne externes OMEdit.


In [3]:
import os
import sys
import shutil
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    """Repo-Wurzel finden: enthält 'Inhalt/simple_reservoir/BlueRiver2'."""
    for p in [start, *start.parents]:
        if (p / "Inhalt" / "simple_reservoir" / "BlueRiver2").exists():
            return p
    raise FileNotFoundError(
        "Repository-Wurzel mit Inhalt/simple_reservoir/BlueRiver2 nicht gefunden."
    )

REPO_ROOT     = find_repo_root(Path.cwd())
BLUERIVER_DIR = REPO_ROOT / "Inhalt" / "simple_reservoir" / "BlueRiver2"
DATA_ROOT     = REPO_ROOT / "Inhalt" / "Notebook_Daten" / "RTC_BlueRiver"

SCENARIO_SHORT = DATA_ROOT / "ShortTerm" / "input"
SCENARIO_LONG  = DATA_ROOT / "LongTerm"  / "input"
DATA_MODEL     = DATA_ROOT / "model"

CASE_DIR    = DATA_ROOT / "runtime_case"
CASE_INPUT  = CASE_DIR / "input"
CASE_MODEL  = CASE_DIR / "model"
CASE_SRC    = CASE_DIR / "src"
CASE_XSD    = CASE_DIR / "xsd"
CASE_OUTPUT = CASE_DIR / "output"

# Statt eines separaten Sub-venv läuft das Modell direkt im Kernel-Interpreter.
PY = sys.executable

print(f"Repo root:     {REPO_ROOT}")
print(f"BlueRiver:     {BLUERIVER_DIR}")
print(f"Data root:     {DATA_ROOT}")
print(f"Runtime case:  {CASE_DIR}")
print(f"Python:        {PY}")


Repo root:     c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel
BlueRiver:     c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2
Data root:     c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver
Runtime case:  c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Python:        c:\Users\grego\anaconda3\envs\hausarbeit_wb4\python.exe


## Pre-Run Checks

Vor dem Lauf prüfen wir, ob alle benötigten Modell- und Szenariodateien vorhanden sind.


In [4]:
required_blue = [
    BLUERIVER_DIR / "src"   / "BlueRiver.py",
    BLUERIVER_DIR / "model" / "BlueRiver.mo",
    BLUERIVER_DIR / "xsd"   / "rtcDataConfig.xsd",
    BLUERIVER_DIR / "xsd"   / "rtcSharedTypes.xsd",
]

required_data = [
    DATA_MODEL / "reservoirs.csv",
    DATA_MODEL / "volumelevel.csv",
]

required_short = [
    SCENARIO_SHORT / "goal_table.csv",
    SCENARIO_SHORT / "plot_table.csv",
    SCENARIO_SHORT / "rtcDataConfig.xml",
    SCENARIO_SHORT / "rtcParameterConfig.xml",
    SCENARIO_SHORT / "timeseries_import.csv",
    SCENARIO_SHORT / "timeseries_import.xml",
]

missing = [p for p in (required_blue + required_data + required_short) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Folgende Dateien fehlen:\n" + "\n".join(str(p) for p in missing)
    )

print("Alle Pflichtdateien vorhanden.")
print(f"BlueRiver.py:  {required_blue[0]}")
print(f"BlueRiver.mo:  {required_blue[1]}")


Alle Pflichtdateien vorhanden.
BlueRiver.py:  c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2\src\BlueRiver.py
BlueRiver.mo:  c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2\model\BlueRiver.mo


## Runtime-Case vorbereiten

Wir erzeugen eine lauffähige Case-Struktur in `Inhalt/Notebook_Daten/RTC_BlueRiver/runtime_case`.
Dadurch landen alle Ergebnisse automatisch im `Notebook_Daten`-Ordner.


In [5]:
import shutil
import stat
import time

def _on_rm_error(func, path, exc_info):
    # Windows/OneDrive: read-only Attribute entfernen und erneut versuchen
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception:
        pass

def _remove_path(path: Path):
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path, onerror=_on_rm_error)
    else:
        try:
            os.chmod(path, stat.S_IWRITE)
        except Exception:
            pass
        path.unlink(missing_ok=True)

def _clear_dir(path: Path, retries: int = 5, delay_s: float = 0.4):
    path.mkdir(parents=True, exist_ok=True)
    for child in list(path.iterdir()):
        for attempt in range(1, retries + 1):
            try:
                _remove_path(child)
                break
            except PermissionError:
                if attempt == retries:
                    raise
                time.sleep(delay_s * attempt)

def prepare_case(scenario_input: Path):
    # Frische Ordnerstruktur (Output bleibt erhalten, damit figures/<RUN_TAG>/
    # zwischen Konfigurationen nicht überschrieben werden).
    for p in [CASE_INPUT, CASE_MODEL, CASE_SRC, CASE_XSD]:
        _clear_dir(p)
    CASE_OUTPUT.mkdir(parents=True, exist_ok=True)

    shutil.copytree(BLUERIVER_DIR / "src",   CASE_SRC,   dirs_exist_ok=True)
    shutil.copytree(BLUERIVER_DIR / "model", CASE_MODEL, dirs_exist_ok=True)
    shutil.copytree(BLUERIVER_DIR / "xsd",   CASE_XSD,   dirs_exist_ok=True)

    shutil.copy2(DATA_MODEL / "reservoirs.csv",  CASE_MODEL / "reservoirs.csv")
    shutil.copy2(DATA_MODEL / "volumelevel.csv", CASE_MODEL / "volumelevel.csv")

    shutil.copytree(scenario_input, CASE_INPUT, dirs_exist_ok=True)

    print("Case vorbereitet:", CASE_DIR)
    print("Input:",  CASE_INPUT)
    print("Output:", CASE_OUTPUT)


## ShortTerm-Szenario aktivieren

Wir nutzen das ShortTerm-Szenario aus `Inhalt/Notebook_Daten/RTC_BlueRiver/ShortTerm/input` und kopieren es in den Runtime-Case.

> **Hinweis:** Diese Zelle setzt die Eingabedateien (inkl. `goal_table.csv`) auf den ShortTerm-Default zurück. Wenn Sie zwischendurch Goals editiert haben (siehe nächste Zelle), gehen die Änderungen verloren — führen Sie diese Zelle also nur aus, wenn Sie *bewusst* zurücksetzen möchten.


In [6]:
prepare_case(SCENARIO_SHORT)
print("ShortTerm-Szenario ist aktiv.")


Case vorbereitet: c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Input: c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\input
Output: c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output
ShortTerm-Szenario ist aktiv.


## Interaktiver Goal-Editor (Bearbeiten und Speichern)

Diese GUI dient ausschließlich dazu, die Ziel-Tabelle `runtime_case/input/goal_table.csv` zu bearbeiten und zu **speichern**. Der Workflow ist bewusst sequentiell:

1. Werte in der GUI ändern (active / priority / weight / order pro Goal).
2. **Speichern** klicken — die Datei wird überschrieben.
3. In den folgenden Zellen das Modell ausführen, Ergebnisse plotten und (optional) die Plots als PNG/HTML exportieren.

So können Sie verschiedene Ziel-Konfigurationen gegeneinander vergleichen, indem Sie für jede Konfiguration einen eigenen `RUN_TAG` setzen und die Bilder im `output/figures/`-Ordner ablegen.


In [7]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

GOAL_TABLE_RUNTIME = CASE_INPUT / "goal_table.csv"

if not GOAL_TABLE_RUNTIME.exists():
    raise FileNotFoundError(f"Fehlende Datei: {GOAL_TABLE_RUNTIME}")

_editor_df = pd.read_csv(GOAL_TABLE_RUNTIME)
_row_controls = {}


def _bool_from_cell(v):
    if pd.isna(v):
        return False
    if isinstance(v, str):
        return v.strip().lower() in {"1", "true", "yes", "y"}
    return bool(int(v)) if isinstance(v, (int, float)) else bool(v)


def _int_or_default(v, default=0):
    try:
        return int(float(v))
    except Exception:
        return default


def _float_or_default(v, default=1.0):
    try:
        return float(v)
    except Exception:
        return default


def _build_row(idx, row):
    id_val = _int_or_default(row.get("id"), idx)
    state = str(row.get("state", ""))
    goal_type = str(row.get("goal_type", ""))

    meta = widgets.HTML(value=f"<b>ID {id_val}</b> | {state} | {goal_type}",
                        layout=widgets.Layout(width="360px"))
    active = widgets.Checkbox(value=_bool_from_cell(row.get("active", 0)),
                              description="active", indent=False,
                              layout=widgets.Layout(width="90px"))
    priority = widgets.BoundedIntText(value=_int_or_default(row.get("priority"), 1),
                                      min=0, max=100, description="prio",
                                      layout=widgets.Layout(width="140px"))
    weight = widgets.FloatText(value=_float_or_default(row.get("weight"), 1.0),
                               description="weight",
                               layout=widgets.Layout(width="160px"))
    order = widgets.BoundedIntText(value=max(1, _int_or_default(row.get("order"), 1)),
                                   min=1, max=10, description="order",
                                   layout=widgets.Layout(width="150px"))

    box = widgets.HBox([meta, active, priority, weight, order])
    _row_controls[idx] = {"active": active, "priority": priority,
                          "weight": weight, "order": order}
    return box


def _collect_df_from_controls(base_df):
    df = base_df.copy()
    for idx in df.index:
        c = _row_controls[idx]
        df.at[idx, "active"]   = 1 if c["active"].value else 0
        df.at[idx, "priority"] = int(c["priority"].value)
        df.at[idx, "weight"]   = float(c["weight"].value)
        df.at[idx, "order"]    = int(c["order"].value)
    return df


status = widgets.Output(layout=widgets.Layout(border="1px solid #ddd",
                                              padding="6px",
                                              max_height="160px",
                                              overflow="auto"))

save_btn = widgets.Button(description="Goals speichern", button_style="success",
                          icon="save",
                          layout=widgets.Layout(width="220px"))


def _on_save(_):
    base_df = pd.read_csv(GOAL_TABLE_RUNTIME)
    new_df = _collect_df_from_controls(base_df)
    new_df.to_csv(GOAL_TABLE_RUNTIME, index=False)
    with status:
        clear_output(wait=True)
        print(f"Gespeichert nach: {GOAL_TABLE_RUNTIME}")
        print()
        print(new_df[["id", "state", "active", "priority", "weight", "order"]].to_string(index=False))


save_btn.on_click(_on_save)

rows = [_build_row(idx, row) for idx, row in _editor_df.iterrows()]

display(widgets.VBox([
    widgets.HTML(
        "<b>Goal-Editor</b> — Werte anpassen und auf <i>Goals speichern</i> klicken. "
        f"Die Datei wird unter <code>{GOAL_TABLE_RUNTIME}</code> abgelegt."
    ),
    widgets.VBox(rows),
    save_btn,
    status,
]))

with status:
    clear_output(wait=True)
    print(f"Editor bereit. Aktuelle Datei: {GOAL_TABLE_RUNTIME}")


## BlueRiver ausführen

Diese Zelle startet die Optimierung **einmalig**. Sie ist der einzige Punkt, an dem der Solver läuft — die GUI weiter oben hat *nur* die Aufgabe, die Goal-Tabelle zu speichern.

Vergeben Sie unten ein `RUN_TAG`, um diese Konfiguration zu benennen — der Tag wird beim Export der Plots als Ordnername verwendet (siehe Zelle weiter unten). So lassen sich verschiedene Goal-Konfigurationen sauber gegenüberstellen.

> **Tipp:** Setzen Sie sprechende Tags wie `"shortterm_default"`, `"shortterm_high_flood_priority"`, `"shortterm_recreation_off"` — der Tag landet 1:1 als Ordnername im Output.


In [11]:
import subprocess

# Tag für diesen Optimierungslauf — wird später als Ordnername für die exportierten
# Bilder verwendet. Ändern Sie den Tag, bevor Sie die Goal-Tabelle modifizieren
# und neu rechnen, um die Ergebnisse vergleichbar abzulegen.
RUN_TAG = "shortterm_default"

run_cmd = [PY, "BlueRiver.py"]
print(f"RUN_TAG: {RUN_TAG}")
print("Starte:", " ".join(run_cmd))

run_env = os.environ.copy()
run_env["MPLBACKEND"] = "Agg"
result = subprocess.run(run_cmd, cwd=CASE_SRC, env=run_env,
                        capture_output=True, text=True)
combined_log = (result.stdout or "") + "\n" + (result.stderr or "")
log_path = CASE_DIR / "venv-log.txt"
log_path.write_text(combined_log, encoding="utf-8")

print("Return code:", result.returncode)
print("Log gespeichert in:", log_path)
print("\n--- Letzte Logzeilen ---")
print("\n".join(combined_log.splitlines()[-30:]))

if result.returncode != 0:
    raise RuntimeError(
        "BlueRiver-Lauf fehlgeschlagen. Siehe venv-log.txt und Troubleshooting-Abschnitt."
    )


RUN_TAG: shortterm_default
Starte: c:\Users\grego\anaconda3\envs\hausarbeit_wb4\python.exe BlueRiver.py
Return code: 0
Log gespeichert in: c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\venv-log.txt

--- Letzte Logzeilen ---
2026-04-20 16:12:03,897 INFO Mapping
2026-04-20 16:12:03,897 INFO Composing NLP segment
2026-04-20 16:12:03,897 INFO Creating NLP dictionary
2026-04-20 16:12:03,897 INFO Done transcribing problem
CasADi - 2026-04-20 16:12:03 WARNING("CasADi was not compiled with WITH_OPENMP=ON. Falling back to serial evaluation.") [.../casadi/core/map.cpp:406]
CasADi - 2026-04-20 16:12:03 WARNING("CasADi was not compiled with WITH_OPENMP=ON. Falling back to serial evaluation.") [.../casadi/core/map.cpp:406]
2026-04-20 16:12:03,912 INFO Calling solver
2026-04-20 16:12:03,980 INFO Solver succeeded with status Solve_Succeeded (0.078253 seconds).
2026-04-20 16:12:03,980 INFO Done

## Ergebnisdateien validieren

Wir prüfen, ob die Exportdateien geschrieben wurden und ob die Kerndaten vorhanden sind.


In [12]:
import xml.etree.ElementTree as ET
import pandas as pd

out_csv = CASE_OUTPUT / "timeseries_export.csv"
out_diag = CASE_OUTPUT / "diag.xml"
perf_dir = CASE_OUTPUT / "performance_metrics"
run_log = CASE_DIR / "venv-log.txt"

if not out_csv.exists():
    raise FileNotFoundError(f"Fehlt: {out_csv}")

df = pd.read_csv(out_csv, parse_dates=["time"])
required_cols = ["time", "TroutLake_V", "TroutLake_Q_out", "RiverCity_Q"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError("Fehlende Spalten in timeseries_export.csv: " + ", ".join(missing_cols))
if df.empty:
    raise ValueError("timeseries_export.csv ist leer.")

diag_info = "Keine Diag-Information gefunden."
if out_diag.exists():
    root = ET.parse(out_diag).getroot()
    lines = list(root)
    levels = {"1": 0, "2": 0, "3": 0}
    for line in lines:
        lvl = line.attrib.get("level")
        if lvl in levels:
            levels[lvl] += 1
    diag_info = f"diag.xml vorhanden. Level-Count: {levels}"
elif perf_dir.exists() and list(perf_dir.glob("*.csv")):
    diag_info = f"diag.xml fehlt, aber performance_metrics vorhanden ({len(list(perf_dir.glob('*.csv')))} Dateien)."
elif run_log.exists():
    log_text = run_log.read_text(encoding="utf-8", errors="ignore")
    if "Traceback" in log_text:
        raise RuntimeError("venv-log.txt enthaelt einen Traceback. Lauf pruefen.")
    if "Done goal programming" in log_text:
        diag_info = "diag.xml fehlt, aber Lauf laut venv-log.txt erfolgreich (Done goal programming)."
    else:
        raise FileNotFoundError("Weder diag.xml noch performance_metrics gefunden. Laufpruefung unvollstaendig.")
else:
    raise FileNotFoundError("Weder diag.xml noch performance_metrics/venv-log.txt gefunden.")

print("CSV ok. Zeilen:", len(df))
print(diag_info)
short_term_df = df.copy()


CSV ok. Zeilen: 57
diag.xml fehlt, aber performance_metrics vorhanden (4 Dateien).


## Ergebnisse plotten (Plotly, alle Prioritäten)

Wir lesen das aktuellste `cached_results/*.json` aus dem Optimierungslauf und erzeugen für **jede** Priorität die passende Goal-Visualisierung. Die Figuren landen im Dictionary `figures` (Schlüssel = stabiler Dateiname-Stamm), damit die nächste Zelle sie als PNG/HTML exportieren kann.


In [ ]:
# Plotly-Renderer auf CDN-basiert setzen — sonst landet plotly.js (~3 MB)
# pro Figur INLINE in der gerenderten HTML-Seite (Jupyter-Book-Build).
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import json
from collections import OrderedDict

import pandas as pd
import plotly.graph_objects as go


CACHE_DIR = CASE_OUTPUT / "cached_results"
cache_files = sorted(CACHE_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime)
if not cache_files:
    raise FileNotFoundError(f"Keine cached_results-Datei gefunden in: {CACHE_DIR}")

cache_path = cache_files[-1]
cache_data = json.loads(cache_path.read_text(encoding="utf-8"))

intermediate_results = cache_data.get("intermediate_results", [])
if not intermediate_results:
    raise ValueError("cached_results enthält keine intermediate_results.")


def _unwrap_array(value):
    if isinstance(value, dict) and "data" in value:
        return value["data"]
    return value


io_datetimes = cache_data["prio_independent_data"]["io_datetimes"]
time_index = pd.to_datetime([item["value"] for item in io_datetimes])

base_goals = {
    int(goal["goal_id"]): goal
    for goal in cache_data["prio_independent_data"]["base_goals"]
}
plot_rows = [item["value"] for item in cache_data["plot_options"]["plot_config"]]

color_map = {
    "TroutLake_V":         "royalblue",
    "RiverCity_Q":         "green",
    "TroutLake_Q_out":     "pink",
    "Alder_Inflow":        "olive",
    "TroutLake_Q_spill":   "brown",
    "TroutLake_Q_turbine": "violet",
}


def _add_target(fig, name, target_series):
    if target_series is None:
        return
    values = _unwrap_array(target_series)
    if not values:
        return
    numeric = [float(v) for v in values]
    if all(abs(v - numeric[0]) < 1e-12 for v in numeric):
        fig.add_hline(
            y=numeric[0], line_dash="dash", line_color="red",
            annotation_text=name,
            annotation_position="top right" if "max" in name.lower() else "bottom right",
        )
    else:
        fig.add_trace(go.Scatter(
            x=time_index[: len(numeric)], y=numeric, mode="lines",
            name=name, line=dict(color="red", dash="dash"),
        ))


def _build_figures_for_priority(priority, snapshot):
    """Return an OrderedDict[stem -> Figure] for a single priority snapshot."""
    timeseries_data = {
        key: _unwrap_array(value)
        for key, value in snapshot.get("timeseries_data", {}).items()
    }
    figs = OrderedDict()
    for row in plot_rows:
        goal_id = int(row["id"])
        goal = base_goals.get(goal_id)
        if goal is None:
            continue

        main_var = goal.get("state")
        extra_vars = row.get("variables_style_2", [])
        variables = [main_var] + [v for v in extra_vars if v != main_var]

        fig = go.Figure()
        for var in variables:
            values = timeseries_data.get(var)
            if values is None:
                continue
            y = [float(v) for v in values]
            x = time_index[: len(y)]
            fig.add_trace(go.Scatter(
                x=x, y=y, mode="lines", name=var,
                line=dict(color=color_map.get(var, None)),
            ))

        _add_target(fig, "Target max", goal.get("target_max_series"))
        _add_target(fig, "Target min", goal.get("target_min_series"))

        subtitle = row.get("custom_title", f"Goal {goal_id}")
        title = f"Priority {priority} | {subtitle}"
        fig.update_layout(
            title=title, xaxis_title="Time",
            yaxis_title=str(row.get("y_axis_title", "Value")).replace("$", ""),
            template="plotly_white",
        )

        # Stable filename stem
        safe_state = str(main_var or "var").replace("/", "_").replace(" ", "_")
        figs[f"priority_{priority:02d}__goal_{goal_id:03d}__{safe_state}"] = fig
    return figs


available_priorities = sorted(int(item.get("priority")) for item in intermediate_results)
figures = OrderedDict()
for prio in available_priorities:
    snapshot = next(item for item in intermediate_results if int(item.get("priority")) == prio)
    figures.update(_build_figures_for_priority(prio, snapshot))

print(f"Cached results: {cache_path.name}")
print(f"Verfügbare Prioritäten: {available_priorities}")
print(f"Anzahl erzeugter Figuren: {len(figures)}")

# Inline anzeigen
for fig in figures.values():
    fig.show()


## Optional: LongTerm-Szenario

Diese Erweiterung ist optional. Setzen Sie in der nächsten Zelle `RUN_LONGTERM = True`,
um LongTerm zu aktivieren, erneut zu rechnen und mit ShortTerm zu vergleichen.

> **Achtung:** Diese Zelle ruft intern erneut `prepare_case` auf — die `goal_table.csv` wird damit auf den **LongTerm-Default** gesetzt. Wenn Sie die ShortTerm-Goals weiter editieren möchten, lassen Sie `RUN_LONGTERM = False`.


In [14]:
import plotly.graph_objects as go

RUN_LONGTERM = True

if not RUN_LONGTERM:
    print("LongTerm uebersprungen. Setzen Sie RUN_LONGTERM=True fuer die Erweiterung.")
else:
    prepare_case(SCENARIO_LONG)

    run_env_long = os.environ.copy()
    run_env_long["MPLBACKEND"] = "Agg"
    result_long = subprocess.run([PY, "BlueRiver.py"], cwd=CASE_SRC,
                                 env=run_env_long, capture_output=True, text=True)
    combined_long = (result_long.stdout or "") + "\n" + (result_long.stderr or "")
    (CASE_DIR / "venv-log.txt").write_text(combined_long, encoding="utf-8")
    if result_long.returncode != 0:
        raise RuntimeError("LongTerm-Lauf fehlgeschlagen. Siehe venv-log.txt")

    long_df = pd.read_csv(CASE_OUTPUT / "timeseries_export.csv", parse_dates=["time"])
    print("LongTerm-Zeilen:", len(long_df))

    fig_long_v = go.Figure()
    fig_long_v.add_trace(go.Scatter(x=short_term_df["time"], y=short_term_df["TroutLake_V"],
                                    mode="lines", name="ShortTerm",
                                    line=dict(color="royalblue")))
    fig_long_v.add_trace(go.Scatter(x=long_df["time"], y=long_df["TroutLake_V"],
                                    mode="lines", name="LongTerm",
                                    line=dict(color="orange")))
    fig_long_v.update_layout(title="Vergleich ShortTerm vs LongTerm: TroutLake_V",
                             xaxis_title="Time", yaxis_title="TroutLake_V [m^3]",
                             template="plotly_white")
    fig_long_v.show()

    fig_long_q = go.Figure()
    fig_long_q.add_trace(go.Scatter(x=short_term_df["time"], y=short_term_df["TroutLake_Q_out"],
                                    mode="lines", name="ShortTerm",
                                    line=dict(color="green")))
    fig_long_q.add_trace(go.Scatter(x=long_df["time"], y=long_df["TroutLake_Q_out"],
                                    mode="lines", name="LongTerm",
                                    line=dict(color="red")))
    fig_long_q.update_layout(title="Vergleich ShortTerm vs LongTerm: TroutLake_Q_out",
                             xaxis_title="Time", yaxis_title="TroutLake_Q_out [m^3/s]",
                             template="plotly_white")
    fig_long_q.show()

    # Auch die LongTerm-Vergleichsfiguren in den figures/<RUN_TAG>/-Ordner exportieren.
    long_dir = CASE_OUTPUT / "figures" / f"{RUN_TAG}_longterm_compare"
    long_dir.mkdir(parents=True, exist_ok=True)
    fig_long_v.write_html(str(long_dir / "compare__TroutLake_V.html"),
                          include_plotlyjs="cdn")
    fig_long_q.write_html(str(long_dir / "compare__TroutLake_Q_out.html"),
                          include_plotlyjs="cdn")
    print(f"LongTerm-Vergleichsfiguren als HTML exportiert nach: {long_dir}")


Case vorbereitet: c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Input: c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\input
Output: c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output
LongTerm-Zeilen: 61


LongTerm-Vergleichsfiguren als HTML exportiert nach: c:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output\figures\shortterm_default_longterm_compare


## Troubleshooting

1. **`ModuleNotFoundError` für RTC-Pakete**
   - Prüfen, dass die aktive Umgebung über `pip install -r requirements.txt` aktualisiert wurde.
   - In der Smoke-Check-Zelle ganz oben wird angezeigt, welche Pakete fehlen.

2. **OpenModelica wird nicht gefunden** *(nur relevant für Editieren in OMEdit)*
   - Für das *Ausführen* dieses Notebooks ist OpenModelica nicht erforderlich — die Modellkompilierung übernimmt `pymoca`/`casadi` direkt.
   - Wenn Sie das Modell visuell bearbeiten möchten: OpenModelica installieren (siehe Hinweis oben).

3. **Pfadprobleme (z.B. OneDrive, Leerzeichen)**
   - Notebook immer aus dem Repository-Kontext starten (Jupyter-Server in der Repo-Wurzel).
   - Keine relativen Pfade manuell ändern; die Zellen verwenden `REPO_ROOT` und `DATA_ROOT`.

4. **Warnungen/Fehler in Diagnoseausgaben**
   - Primär `Inhalt/Notebook_Daten/RTC_BlueRiver/runtime_case/venv-log.txt` prüfen.
   - Falls vorhanden, `output/diag.xml` auswerten (Level 1 = Fehler).
   - Alternativ `output/performance_metrics/*.csv` als Laufnachweis nutzen.
   - Bei Infeasibility zuerst Szenario-Input und Goal-Tabellen prüfen.

5. **Permission denied beim Aufräumen des Runtime-Cases (Windows/OneDrive)**
   - Manchmal sperrt OneDrive Dateien während der Synchronisation. Notebook-Kernel neu starten und erneut ausführen.

6. **PNG-Export per `kaleido` hängt / wird übersprungen**
   - `kaleido` startet intern einen Chromium-Subprozess. Auf einigen Windows-Systemen (insb. mit OneDrive oder restriktivem AV) startet dieser nicht innerhalb der 30s-Frist. Die HTML-Versionen werden trotzdem geschrieben — diese sind ohnehin interaktiver und für den meisten Berichts-Workflow ausreichend.
   - Workarounds: (a) `kaleido` neu installieren mit `pip install --force-reinstall 'kaleido==0.2.1'`, (b) auf Linux/Binder ausführen — dort funktioniert PNG-Export zuverlässig, (c) `EXPORT_PNG = False` setzen und die HTML-Datei mit dem Browser-Druckdialog als PDF/Bild exportieren.
